In [ ]:
# 1. Install Unsloth and requirements
!pip install unsloth 
import torch
from unsloth import FastLanguageModel


In [ ]:

# 2. Load Llama 3.2 3B (matching your local version)
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Llama-3.2-3B-Instruct",
    max_seq_length = 2048,
    load_in_4bit = True,
)

# 3. Add LoRA Adapters (Rank 16 to stay ~200MB)
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, 
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
)

In [ ]:
from datasets import load_dataset
import json

# Load your uploaded JSONL file
dataset = load_dataset("json", data_files={"train": "vijay-profile-for-finetuning.jsonl"}, split="train")

# Define how the model should learn about you
def format_vijay_profile(example):
    # This creates a pair where the "user" asks about Vijay and the "assistant" answers using the JSON data
    # In a real scenario, you'd have multiple Q&A pairs. For a single profile, we treat it as a summary.
    formatted_text = f"<|begin_of_text|><|start_header_id|>user<|end_header_id|>\n\nWho is Vijay Rajesh R?<|eot_id|>" \
                     f"<|start_header_id|>assistant<|end_header_id|>\n\n" \
                     f"Vijay Rajesh R is an {example['personal_info']['summary']}. " \
                     f"He is currently pursuing {example['education'][0]['degree']} in {example['education'][0]['major']} at {example['education'][0]['institution']}. " \
                     f"His technical skills include {', '.join(example['technical_skills']['artificial_intelligence'])}.<|eot_id|>"
    return {"text": formatted_text}

dataset = dataset.map(format_vijay_profile)

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = 2048,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        max_steps = 60, # Small dataset needs very few steps
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        output_dir = "VIJAY_LLM_outputs",
    ),
)
trainer.train()

# Export only the LoRA adapters
model.save_pretrained("vijay_lora_adapter")
tokenizer.save_pretrained("vijay_lora_adapter")

# HOW TO USE THE ADAPTER
-------------

The Essential Files
adapter_model.safetensors: This is the most important file. It contains the actual "learned" weights from your profile.

adapter_config.json: This tells Ollama how to apply those weights to the base Llama 3.2 model.

Note: You do not need the .pt or .pth files (like optimizer.pt or rng_state.pth). Those are only used if you want to resume training later in Colab.

----------------

On your local machine, where you have llama3.2:latest:Create a Modelfile in the same folder as your downloaded .safetensors file:Dockerfile# Specify the local base model you already have
FROM llama3.2

# Point to the adapter you downloaded from Colab
ADAPTER ./adapter_model.safetensors

# Set your custom system prompt
SYSTEM "You are an AI assistant that knows everything about Vijay Rajesh R's professional profile."
Build and Run in your terminal:Bashollama create vijay-gpt -f Modelfile
ollama run vijay-gpt


#### Why this fits under 200 MB limitBy setting the Rank to 16, the resulting adapter_model.safetensors will be approximately 170 MB. This allows Ollama to load the base 2.0 GB llama3.2:latest model and "patch" it with your specific profile data instantly during inference.